# Лабораторна робота №3

In [9]:
N = ord("S") % 3 + 1
print(N)

3


### Варіант 3

Створити власний декоратор, для кешування результатів функції – якщо 
певна функція вже запускалась с такими параметрами – не запускати 
функцію, а відобразити результати із кешу.  Протестувати на 3-4 функціях з 
різною кількістю та типом параметрів. 

In [4]:
import functools
import time

def make_hashable(obj):          #паеретворюємо списки та множини у незмінювані кортеджи
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    if isinstance(obj, (list, tuple, set)):
        return tuple(make_hashable(x) for x in obj)
    return obj

def cache_decorator(func):
    cache = {}

    @functools.wraps(func)
    def wrapper(*args, **kwargs):     #беремо звичайні аршуманти та іменовані
        key = (
            tuple(make_hashable(a) for a in args),
            tuple(sorted((k, make_hashable(v)) for k, v in kwargs.items())),
        )
        if key in cache:
            print(f"[CACHE HIT]  {func.__name__}{args}{kwargs}")
            return cache[key]

        print(f"[CALCULATE]  {func.__name__}{args}{kwargs}")
        result = func(*args, **kwargs)
        cache[key] = result
        return result

    wrapper.cache = cache
    return wrapper


# --- Визначення функцій ---
@cache_decorator
def square(n):
    time.sleep(1)
    return n * n


@cache_decorator
def add(a, b):
    time.sleep(1)
    return a + b


@cache_decorator
def greet(name, greeting="Hello"):
    time.sleep(1)
    return f"{greeting}, {name}!"


@cache_decorator
def sum_list(numbers):
    time.sleep(1)
    return sum(numbers)


@cache_decorator
def stats(data, factor=1):
    time.sleep(1)
    return {"sum": sum(data) * factor, "count": len(data)}

#### ТЕСТУВАННЯ

In [5]:
print(square(5))
print(square(5))                # HIT
print(square(7))                # нові аргументи

print(add(2, 3))
print(add(2, 3))                # HIT
print(add(a=2, b=3))
print(add(a=2, b=3))            # HIT

print(greet("Ivan"))
print(greet("Ivan"))
print(greet("Ivan", greeting="Hi"))
print(greet("Ivan", greeting="Hi"))   # HIT

print(sum_list([1, 2, 3]))      #
print(sum_list([1, 2, 3]))      # HIT

print(stats([1, 2, 3], factor=2))
print(stats([1, 2, 3], factor=2))     # HIT

[CALCULATE]  square(5,){}
25
[CACHE HIT]  square(5,){}
25
[CALCULATE]  square(7,){}
49
[CALCULATE]  add(2, 3){}
5
[CACHE HIT]  add(2, 3){}
5
[CALCULATE]  add(){'a': 2, 'b': 3}
5
[CACHE HIT]  add(){'a': 2, 'b': 3}
5
[CALCULATE]  greet('Ivan',){}
Hello, Ivan!
[CACHE HIT]  greet('Ivan',){}
Hello, Ivan!
[CALCULATE]  greet('Ivan',){'greeting': 'Hi'}
Hi, Ivan!
[CACHE HIT]  greet('Ivan',){'greeting': 'Hi'}
Hi, Ivan!
[CALCULATE]  sum_list([1, 2, 3],){}
6
[CACHE HIT]  sum_list([1, 2, 3],){}
6
[CALCULATE]  stats([1, 2, 3],){'factor': 2}
{'sum': 12, 'count': 3}
[CACHE HIT]  stats([1, 2, 3],){'factor': 2}
{'sum': 12, 'count': 3}
